# Day 13 · 告別失控的 AI：Graph Workflows 與 Graph Routes

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 13 - 告別失控的 AI：Graph Workflows 與 Graph Routes.md`

> 📌 **建議先讀 [Day 16](../day16_template_workflow_agents/)**：
> Sequential / Parallel / Loop 是跨語言共通的原語，也是理解圖的墊腳石。

## 今天要學會

1. 用 `Workflow` + `edges` 畫出一張執行圖
2. 用 `ctx.route` 做**條件分支**
3. 用 `JoinNode` 做 fan-out / join

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 為什麼 prompt 寫再仔細都不夠

「金額超過一萬要主管簽核，否則直接放行。」

這種再普通不過的需求，用 prompt 寫是這樣的：

> 「請判斷金額，如果超過一萬就……否則就……」

問題是**模型只是在猜下一個字**。它大部分時候會對，但你無法保證。
而且你沒辦法測試「金額 10001 時一定會走簽核」這件事。

ADK 2.0 的 `Workflow` 讓分支變成**確定性的 Python 程式碼**。

## 2. 圖的三個要素

| 概念 | 是什麼 |
|---|---|
| **節點** | 一個 `LlmAgent`，或一個 `@node` 裝飾的函式 |
| **邊** | `(從哪, 到哪)` 的 tuple |
| **`START`** | 圖的入口，使用者訊息從這裡進來 |

In [2]:
from google.adk import Workflow
from google.adk.agents import LlmAgent
from google.adk.agents.context import Context
from google.adk.runners import InMemoryRunner
from google.adk.workflow import START, JoinNode, node
from google.genai import types
from pydantic import BaseModel, Field


async def run_graph(workflow, text: str, *, show=True):
    """跑一張圖，印出每個節點的 output，回傳最後一個。"""
    runner = InMemoryRunner(agent=workflow, app_name="day13")
    sid = await new_session(runner)
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    last = None
    async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
        out = getattr(ev, "output", None)
        if out is not None:
            if show:
                print(f"  ▪ {out}")
            last = out
        elif ev.is_final_response() and ev.content:
            t = "".join(p.text or "" for p in ev.content.parts if p.text)
            if t and show:
                print(f"  💬 {t.strip()[:120]}")
    return last

### 最小的圖：三個節點串起來

In [3]:
@node
def normalize(raw_text: str = "") -> str:
    return raw_text.strip().replace("　", " ")


extractor = LlmAgent(
    name="extractor",
    model=get_model(),
    instruction="從使用者的報帳描述中抽出金額（新台幣整數）與用途。",
    output_key="expense",
)


@node
def stamp(expense: dict) -> str:
    return f"已受理：{expense.get('purpose', '未填')} / NT${expense.get('amount', 0)}"


# 這裡只是先看「圖長什麼樣」，下一節才加分支
simple = Workflow(
    name="simple_flow",
    edges=[(START, extractor), (extractor, stamp)],
)
print("邊的數量:", len(simple.edges))

邊的數量: 2


### ⚠️ 節點的參數是從 **state** 綁進來的

這是文章沒講清楚、但寫錯圖就不會動的第一件事。

`@node` 函式的參數**預設從 session state 取值**，
**不是**從上一個節點的回傳值。

In [4]:
import inspect

print("node 裝飾器的簽章：")
print(" ", inspect.signature(node))
print()
print("→ 注意 parameter_binding 預設是 'state'。")
print("  所以上游的 LlmAgent 一定要設 output_key，下游節點才拿得到。")

node 裝飾器的簽章：
  (node_like: 'definitions.NodeLike | None' = None, *, name: 'str | None' = None, rerun_on_resume: 'bool | None' = None, retry_config: 'RetryConfig | None' = None, timeout: 'float | None' = None, parallel_worker: 'bool' = False, max_parallel_workers: 'int | None' = None, auth_config: 'AuthConfig | None' = None, parameter_binding: "Literal['state', 'node_input']" = 'state') -> 'Any'

→ 注意 parameter_binding 預設是 'state'。
  所以上游的 LlmAgent 一定要設 output_key，下游節點才拿得到。


忘了設 `output_key` 的話，會得到這個錯誤：

```
ValueError: Missing value for parameter "expense" of function "stamp".
            It was not found in state and has no default value.
```

這也是為什麼上面 `extractor` 一定要有 `output_key="expense"`。

## 3. 條件分支：`ctx.route`

這是本日的重點，也是 Day 16 那三個 workflow agent **做不到**的事。

規則有三條：

1. 分流節點多收一個 `ctx: Context` 參數
2. 在裡面設 `ctx.route = "某個值"`
3. 邊用 **dict** 表示：`(分流節點, {"值A": 節點A, "值B": 節點B})`

**⚠️ 路由靠的是 `ctx.route`，不是函式的回傳值。** 這是第二個寫錯就不會動的點。

In [5]:
class Expense(BaseModel):
    amount: int = Field(description="金額，新台幣整數")
    purpose: str = Field(description="用途，十個字以內")
    category: str = Field(description="分類：差旅／設備／餐飲／其他")


parser = LlmAgent(
    name="parser",
    model=get_model(),
    instruction="從使用者的報帳描述中抽出金額、用途與分類。",
    output_schema=Expense,
    output_key="expense",
)


@node
def route_by_amount(ctx: Context, expense: dict) -> str:
    """依金額決定審核層級——這是確定性的 Python，不是模型的判斷。"""
    amount = int(expense["amount"])
    if amount >= 50000:
        ctx.route = "director"
    elif amount >= 10000:
        ctx.route = "manager"
    else:
        ctx.route = "auto"
    return f"金額 NT${amount:,} → {ctx.route}"


@node
def auto_approve(expense: dict) -> str:
    return f"✅ 自動核准：{expense['purpose']} NT${expense['amount']:,}"


@node
def need_manager(expense: dict) -> str:
    return f"📋 送主管簽核：{expense['purpose']} NT${expense['amount']:,}"


@node
def need_director(expense: dict) -> str:
    return f"🔴 送處長簽核（高額）：{expense['purpose']} NT${expense['amount']:,}"


expense_flow = Workflow(
    name="expense_flow",
    edges=[
        (START, parser),
        (parser, route_by_amount),
        (route_by_amount, {
            "auto": auto_approve,
            "manager": need_manager,
            "director": need_director,
        }),
    ],
)
print("✅ 圖建立成功")

✅ 圖建立成功


In [6]:
CASES = [
    "我上週搭高鐵去台中出差，車票 1490 元。",
    "買了一台新的開發用筆電，含稅 42800 元。",
    "部門年度教育訓練場地與講師費，總共 88000 元。",
]

for text in CASES:
    print(f"\n📥 {text}")
    await run_graph(expense_flow, text)


📥 我上週搭高鐵去台中出差，車票 1490 元。


  💬 {
  "amount": 1490,
  "purpose": "高鐵車票",
  "category": "差旅"
}
  ▪ 金額 NT$1,490 → auto
  ▪ ✅ 自動核准：高鐵車票 NT$1,490

📥 買了一台新的開發用筆電，含稅 42800 元。


  💬 {
  "amount": 42800,
  "purpose": "買開發筆電",
  "category": "設備"
}
  ▪ 金額 NT$42,800 → manager
  ▪ 📋 送主管簽核：買開發筆電 NT$42,800

📥 部門年度教育訓練場地與講師費，總共 88000 元。


  💬 {
  "amount": 88000,
  "purpose": "教育訓練",
  "category": "其他"
}
  ▪ 金額 NT$88,000 → director
  ▪ 🔴 送處長簽核（高額）：教育訓練 NT$88,000


**同一張圖，三條不同的路徑。**

而且分支條件是 `if amount >= 50000` 這種確定性程式碼——
你可以為它寫單元測試，可以保證邊界值的行為。
模型只負責「從自然語言抽出金額」這件它擅長的事。

### ⚠️ 圖的輸出在 `event.output`，不在 `event.content`

這是第三個容易卡住的點。從 workflow agent 換過來的人特別容易踩。

In [7]:
runner = InMemoryRunner(agent=expense_flow, app_name="day13")
sid = await new_session(runner)
msg = types.Content(role="user", parts=[types.Part(text=CASES[1])])

node_outputs, final_texts = [], []
async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
    if getattr(ev, "output", None) is not None:
        node_outputs.append(ev.output)
    if ev.is_final_response() and ev.content:
        t = "".join(p.text or "" for p in ev.content.parts if p.text)
        if t:
            final_texts.append(t)

print(f"event.output        抓到 {len(node_outputs)} 筆：")
for o in node_outputs:
    print(f"   ▪ {o}")
print(f"\nis_final_response() 抓到 {len(final_texts)} 筆：")
for t in final_texts:
    print(f"   ▪ {t.strip()[:80]}")
print("\n→ 兩者是不同的東西。只看 is_final_response() 會漏掉整張圖的節點輸出。")

event.output        抓到 2 筆：
   ▪ 金額 NT$42,800 → manager
   ▪ 📋 送主管簽核：買開發筆電 NT$42,800

is_final_response() 抓到 1 筆：
   ▪ {
  "amount": 42800,
  "purpose": "買開發筆電",
  "category": "設備"
}

→ 兩者是不同的東西。只看 is_final_response() 會漏掉整張圖的節點輸出。


## 4. Fan-out / Join

`JoinNode` 會**等所有上游節點都跑完**才觸發。

In [8]:
security = LlmAgent(
    name="security_review", model=get_model(),
    description="資安審查",
    instruction="從資安角度審查使用者提出的方案，指出最大的一個風險，一句話，繁體中文。",
    output_key="security",
)
legal = LlmAgent(
    name="legal_review", model=get_model(),
    description="法遵審查",
    instruction="從法遵角度審查使用者提出的方案，指出最大的一個疑慮，一句話，繁體中文。",
    output_key="legal",
)
cost = LlmAgent(
    name="cost_review", model=get_model(),
    description="成本審查",
    instruction="從成本角度審查使用者提出的方案，指出最大的一項支出，一句話，繁體中文。",
    output_key="cost",
)

gather = JoinNode(name="gather")


@node
def summarize(security: str, legal: str, cost: str) -> str:
    return ("三方審查：\n"
            f"  🔒 {security}\n"
            f"  ⚖️  {legal}\n"
            f"  💰 {cost}")


review_flow = Workflow(
    name="review_flow",
    edges=[
        (START, (security, legal, cost)),   # fan-out：一次觸發三個
        (security, gather),
        (legal, gather),
        (cost, gather),                      # join：三個到齊才往下
        (gather, summarize),
    ],
)

print(await run_graph(review_flow, "我們打算把客戶資料放到公有雲的向量資料庫做 RAG。", show=False))

三方審查：
  🔒 最大的資安風險是：未對敏感資料進行去識別化或強加密便直接上傳至公有雲向量資料庫，一旦雲端環境遭駭或遭受提示詞注入攻擊，將導致客戶隱私與機密資料全盤洩漏。
  ⚖️  最大法遵疑慮在於：將未經去識別化的敏感客戶資料上傳至公有雲向量資料庫，恐違反個資法及相關資安法規對資料跨境傳輸、控制權與安全維護義務的要求。
  💰 從成本角度來看，該方案最大的一項支出將會是**隨著客戶資料量與檢索請求暴增而持續累積的向量資料庫雲端儲存與 API 呼叫運算費用**。


## 5. ⚠️ Stuck JoinNode

`JoinNode` 只等「**實際會被觸發**」的上游。
如果某個分支因為條件路由沒被走到，join 就會**一直等下去**。

官方文件稱為 *Stuck JoinNode*。這是圖形工作流最惡名昭彰的坑，
因為它不會報錯——只是永遠不結束。

In [9]:
print("""容易踩到的結構：

        START
          │
      route_node ── ctx.route = "a"（只走 a）
        ╱     ╲
       a       b        ← b 這條路沒被觸發
        ╲     ╱
        JoinNode        ← 永遠等不到 b，卡住
           │
        下游節點

避免方式：
  1. 條件分支之後**不要**直接接 JoinNode
  2. 真的需要的話，讓每條分支都導向同一個匯總節點（不用 Join）
  3. 或是在分支節點用 fan-out tuple 一次觸發所有會被 join 的上游
""")

容易踩到的結構：

        START
          │
      route_node ── ctx.route = "a"（只走 a）
        ╱     ╲
       a       b        ← b 這條路沒被觸發
        ╲     ╱
        JoinNode        ← 永遠等不到 b，卡住
           │
        下游節點

避免方式：
  1. 條件分支之後**不要**直接接 JoinNode
  2. 真的需要的話，讓每條分支都導向同一個匯總節點（不用 Join）
  3. 或是在分支節點用 fan-out tuple 一次觸發所有會被 join 的上游



### 安全的寫法：分支各自導向同一個節點

In [10]:
@node
def merge_point(expense: dict) -> str:
    """三條分支都導到這裡，不用 JoinNode 就不會卡。"""
    return f"📨 已送出審核流程：{expense['purpose']}"


safe_flow = Workflow(
    name="safe_flow",
    edges=[
        (START, parser),
        (parser, route_by_amount),
        (route_by_amount, {
            "auto": auto_approve,
            "manager": need_manager,
            "director": need_director,
        }),
        # 三條分支各自接到同一個節點——沒有 JoinNode，就沒有 stuck 問題
        (auto_approve, merge_point),
        (need_manager, merge_point),
        (need_director, merge_point),
    ],
)

for text in (CASES[0], CASES[2]):
    print(f"\n📥 {text}")
    await run_graph(safe_flow, text)


📥 我上週搭高鐵去台中出差，車票 1490 元。


  💬 {
  "amount": 1490,
  "purpose": "高鐵車票",
  "category": "差旅"
}
  ▪ 金額 NT$1,490 → auto
  ▪ ✅ 自動核准：高鐵車票 NT$1,490
  ▪ 📨 已送出審核流程：高鐵車票

📥 部門年度教育訓練場地與講師費，總共 88000 元。


  💬 {
  "amount": 88000,
  "purpose": "教育訓練",
  "category": "其他"
}
  ▪ 金額 NT$88,000 → director
  ▪ 🔴 送處長簽核（高額）：教育訓練 NT$88,000
  ▪ 📨 已送出審核流程：教育訓練


## 6. 圖 vs Workflow Agent

| | Workflow Agent（Day 16） | Graph Workflow（本日） |
|---|---|---|
| 條件分支 | ❌ 做不到 | ✅ `ctx.route` |
| 平行 + 合流 | 只能整批平行 | ✅ `JoinNode` 細緻控制 |
| 資料傳遞 | `output_key` + `{key?}` | 節點參數**從 state 綁定** |
| 輸出位置 | `event.content` | **`event.output`** |
| 迴圈 | `LoopAgent` | 邊可以指回前面的節點 |
| 版本狀態 | 2.8.0 起 deprecated | 2.0 起的主線 |

**兩套的資料流機制不同**，混用時要清楚自己在用哪一套（Day 14 會完整比較）。

## 7. 其他已知限制

- **一個節點一次執行只會發出一個 `Event.output`**
- **state 不要放大東西**——整張圖共用，塞大檔案會拖垮效能（用 Artifact，Day 11）
- **`Workflow` 目前不能當 `LlmAgent` 的 sub_agent**，兩套編排無法任意巢狀

In [11]:
print("Workflow 可設定的欄位：")
for f in Workflow.model_fields:
    print(f"  {f}")

Workflow 可設定的欄位：
  name
  description
  rerun_on_resume
  wait_for_output
  retry_config
  timeout
  input_schema
  output_schema
  state_schema
  edges
  max_concurrency
  graph


## 8. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `Missing value for parameter "x"` | 上游 agent 忘了設 `output_key`（參數從 **state** 綁定） |
| 分支完全沒發生 | 用 `return` 而不是 **`ctx.route =`** 決定路由 |
| 拿不到節點輸出 | 看錯地方，要看 **`event.output`** 不是 `event.content` |
| 圖跑到一半卡住不結束 | **Stuck JoinNode**——某個上游分支沒被觸發 |
| 圖越跑越慢 | state 塞了大東西，改用 Artifact |
| 想把 Workflow 當 sub_agent | 目前不支援 |

## 9. 動手練習

1. 幫 `expense_flow` 加第四條路 `"reject"`（金額為 0 或負數直接退件）。
2. 把 `route_by_amount` 的 `ctx.route = ...` 那行改成只 `return` 字串，
   重跑，確認分支不會發生。
3. 故意做出一個 Stuck JoinNode（條件分支後接 `JoinNode`），
   用 `Workflow(timeout=15)` 讓它超時而不是永遠卡住。
4. 把 `summarize` 改成一個 `LlmAgent`，觀察節點是 agent 時輸出跑到哪裡去。

## 本日回顧

- **圖把「流程」變成確定性的程式碼**，模型只負責它擅長的判斷。
- **三個要素**：節點（agent 或 `@node`）、邊（tuple）、`START`。
- **⚠️ 節點參數預設從 state 綁定**，所以上游一定要設 `output_key`。
- **⚠️ 條件分支靠 `ctx.route`，不是回傳值**；邊用 dict 表示分支。
- **⚠️ 圖的輸出在 `event.output`**，不是 `event.content`。
- **⚠️ Stuck JoinNode**：條件分支後接 `JoinNode` 會永遠等下去，
  安全做法是讓各分支導向同一個節點。

---
**下一天 → `../day14_data_handling_dynamic_workflows/`**